# Exercise 14: Behavior Cloning for a Reaching Proxy

The textbook exercise asks for pick-and-place. This lightweight notebook uses planar reaching so that the complete pipeline runs on CPU. The same data split, preprocessing, training, and closed-loop evaluation principles transfer to pick-and-place.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from ch13_guide.envs.reaching_env import ReachingEnv, ReachingConfig
from ch13_guide.models import MLPPolicy
from ch13_guide.utils import proportional_controller, set_global_seed, rollout

set_global_seed(7)

In [ ]:
def collect_demonstrations(episodes=300, seed=0):
    env = ReachingEnv(ReachingConfig(max_steps=80))
    observations, actions, episode_ids = [], [], []
    for episode in range(episodes):
        obs, _ = env.reset(seed=seed + episode)
        while True:
            action = proportional_controller(obs)
            observations.append(obs.copy())
            actions.append(action.copy())
            episode_ids.append(episode)
            obs, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
    return np.asarray(observations), np.asarray(actions), np.asarray(episode_ids)

observations, actions, episode_ids = collect_demonstrations()
print(observations.shape, actions.shape)

In [ ]:
# Split by episode, not by adjacent timesteps.
unique_episodes = np.unique(episode_ids)
rng = np.random.default_rng(7)
rng.shuffle(unique_episodes)

n = len(unique_episodes)
train_eps = set(unique_episodes[:int(0.70*n)])
val_eps = set(unique_episodes[int(0.70*n):int(0.85*n)])
test_eps = set(unique_episodes[int(0.85*n):])

train_mask = np.array([e in train_eps for e in episode_ids])
val_mask = np.array([e in val_eps for e in episode_ids])
test_mask = np.array([e in test_eps for e in episode_ids])

obs_mean = observations[train_mask].mean(axis=0)
obs_std = observations[train_mask].std(axis=0) + 1e-6

def normalize_obs(x):
    return (x - obs_mean) / obs_std

train_ds = TensorDataset(
    torch.tensor(normalize_obs(observations[train_mask]), dtype=torch.float32),
    torch.tensor(actions[train_mask], dtype=torch.float32),
)
val_x = torch.tensor(normalize_obs(observations[val_mask]), dtype=torch.float32)
val_y = torch.tensor(actions[val_mask], dtype=torch.float32)
test_x = torch.tensor(normalize_obs(observations[test_mask]), dtype=torch.float32)
test_y = torch.tensor(actions[test_mask], dtype=torch.float32)

loader = DataLoader(train_ds, batch_size=256, shuffle=True)

In [ ]:
model = MLPPolicy()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

for epoch in range(25):
    model.train()
    running = 0.0
    for x, y in loader:
        pred = model(x)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * x.shape[0]

    if epoch % 5 == 0 or epoch == 24:
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(val_x), val_y).item()
        print(f"epoch={epoch:02d} train_mse={running/len(train_ds):.6f} val_mse={val_loss:.6f}")

In [ ]:
model.eval()
with torch.no_grad():
    test_mse = loss_fn(model(test_x), test_y).item()
print("Held-out action MSE:", test_mse)

def learned_policy(obs):
    x = torch.tensor(normalize_obs(obs), dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        return model(x).squeeze(0).numpy()

closed_loop = rollout(
    ReachingEnv(ReachingConfig(max_steps=80)),
    learned_policy,
    episodes=100,
    seed=1000,
)
df = pd.DataFrame(closed_loop)
print(df[["success", "steps", "final_distance", "safety_interventions"]].mean())

## Instructor interpretation

Low held-out MSE does not guarantee high closed-loop success. Ask students to compare both metrics and inspect failed trajectories. A stronger assignment can remove recovery states from the demonstrations, increase observation noise, or alter the test initial-state distribution to expose covariate shift.